# 01 IFC Viewer PoC

Robuster PoC: erkennt PROJECT_ROOT automatisch, lädt standardmäßig `data/sample.ifc` und hebt eine Beispiel-GlobalId hervor.

In [ ]:
from pathlib import Path
from IPython.display import display

from openbim_viewer.ifc_loader import load_ifc, build_ifc_index
from openbim_viewer.viewer_adapter import IFCViewerAdapter

cwd = Path.cwd()
if (cwd / 'src').exists() and (cwd / 'data').exists():
    PROJECT_ROOT = cwd
elif cwd.name == 'notebooks' and (cwd.parent / 'src').exists():
    PROJECT_ROOT = cwd.parent
else:
    candidates = [p for p in [cwd, *cwd.parents] if (p / 'src').exists() and (p / 'data').exists()]
    PROJECT_ROOT = candidates[0] if candidates else cwd

IFC_PATH = PROJECT_ROOT / 'data' / 'sample.ifc'
print(f'PROJECT_ROOT: {PROJECT_ROOT.resolve()}')
print(f'IFC_PATH: {IFC_PATH.resolve()}')

In [ ]:
if not IFC_PATH.exists():
    print('Bitte eine IFC-Datei unter data/sample.ifc ablegen oder IFC_PATH anpassen.')
else:
    ifc_file = load_ifc(IFC_PATH)
    ifc_index = build_ifc_index(ifc_file)
    print(f'IfcProduct im Index: {len(ifc_index)}')

    adapter = IFCViewerAdapter(max_elements=200)
    viewer_widget = adapter.show(IFC_PATH)
    caps = adapter.capabilities()

    print('Viewer Capabilities:')
    print(f'  backend={caps.backend}')
    print(f'  can_render={caps.can_render}')
    print(f'  can_highlight_guid={caps.can_highlight_guid}')

    display(viewer_widget)

    sample_guid = next(iter(ifc_index.keys()), None)
    if sample_guid and caps.can_highlight_guid:
        print(f'Hebe Beispiel-GlobalId hervor: {sample_guid}')
        adapter.highlight_guid(sample_guid)
    elif sample_guid:
        print(f'Beispiel-GlobalId gefunden, aber kein Highlighting verfügbar: {sample_guid}')
    else:
        print('Keine GlobalId im IFC-Index gefunden.')